# 1일차 실습 P3 — 크기 표 채우기 · 어긋난 층 찾기

- 수업 중 이 실습 슬라이드가 나오면 풂 · **맨 위 준비 셀부터** 위에서 아래로 실행
- 셀 실행: 셀을 누르고 **Shift + Enter** 또는 셀 왼쪽 ▶
- Colab 에서 처음 열 때 경고 창이 뜨면 '계속' · 새 파일은 준비 셀에서 데이터를 받느라 몇십 초 걸릴 수 있음
- 빈칸은 `____` · 빈칸을 모두 바꾼 뒤 **그 셀부터 다시 실행**
- 막히면 셀 이름과 **에러 칸 맨 아래 줄**을 채팅으로

## 에러를 읽는 법

- 에러 칸 첫 줄(`...Error    Traceback ...`)은 제목일 뿐 · 무엇이 틀렸는지는 **맨 아래 줄**
- 중간의 `---->` 화살표 줄 · torch 안쪽 칸은 처음엔 건너뜀 · **위에 찍힌 `[안내]` 문장과 맨 아래 줄부터** 읽음

| 맨 아래 줄에 보이는 말 | 뜻 | 먼저 볼 곳 |
|---|---|---|
| `name '____' is not defined` | 빈칸이 남음 | 화살표가 가리키는 줄 |
| `name 'nn'` · `'Xtr'` · `'MyAE'` · `'MyConvAE'` · `'크기_검사'` is not defined | 그 이름을 만든 셀을 안 돌렸거나 런타임이 끊김 | 준비 셀부터 순서대로 다시 |
| `name 'latent_dimm'` · `'Sigmoid'` · `'optimizer'` · `'ae'` is not defined | 내가 친 이름이 틀림 · 철자 · `nn.` 빠짐 · 학습 함수 안 이름은 `model` · `opt` | 화살표가 가리키는 줄 |
| `unexpected indent` · `unindent` | 줄 앞 칸 수가 위아래 줄과 다름 | 바로 위 줄과 같은 칸에서 시작하게 |
| `Perhaps you forgot a comma?` · `'(' was never closed` · `unmatched ')'` | 쉼표나 괄호 | `^` 표시 자리 |
| `mat1 and mat2 shapes cannot be multiplied (AxB and CxD)` | Linear 에 들어온 값 개수 B 와 Linear 첫 숫자 C 가 다름 | `Flatten` 과 `Linear` 첫 숫자 |
| `is not a Module subclass` | 부품 뒤 `()` 가 빠짐 | `nn.ReLU` → `nn.ReLU()` |
| `expected input[...] to have a channels, but got c channels` | 층의 첫 숫자(받는 채널)가 들어온 채널과 다름 | 마지막으로 찍힌 층의 **다음** 층 첫 숫자 |
| `The size of tensor a (..) must match the size of tensor b (..)` | 출력 크기가 정답 크기와 다름 | 층마다 찍힌 모양 · 이름의 대소문자(`x` · `X`) |

- **에러가 없는데** loss 가 0.2 근처에서 안 내려가고 그림이 회색 네모 → 학습이 안 됐을 수 있음 · `backward()` · `step()` 괄호부터 확인

## 준비

In [1]:
import torch
import torch.nn as nn

print("준비 끝 · 이 실습은 데이터를 쓰지 않음")

준비 끝 · 이 실습은 데이터를 쓰지 않음


## P3. 크기 표 채우기 · 어긋난 층 찾기

1. 표의 빈칸(나오는 크기)을 칸 고르기 · 칸과 틈으로 채움
2. **검사 도구** 셀은 고치지 않고 실행만 함 → 그 아래 셀의 `표 = [...]` 에 채운 나오는 크기 앞 세 칸(끝 28 은 이미 적힘)을 옮겨 적고 실행 → 층마다 표와 찍힌 크기를 나란히 보여 줌
3. **처음 달라지는 층**의 설정 한 곳만 고침 → 다시 실행 · 셀이 `통과` 를 찍으면 끝 · 표가 틀리면 표부터 고치라고 알려 줌
   - 고친 층의 설정이 표의 설정 칸과 같아야 함 · 다른 층이나 다른 인자를 바꿔 28 을 만든 것은 통과 아님
- 찍힌 모양 `(1, 8, 14, 14)` = (장 수, 채널, 세로, 가로) · **끝 두 숫자가 크기**
- 막히면 — 힌트: `Conv2d` k3 s2 p1 은 짝수 칸을 딱 반 · `ConvTranspose2d` k3 s2 p1 op1 은 딱 두 배

| 층 | 설정 | 들어가는 크기 | 나오는 크기 |
|---|---|---|---|
| 1번째 Conv2d | kernel 3 · stride 2 · padding 1 | 28 | 14 |
| 2번째 Conv2d | kernel 3 · stride 2 · padding 1 | 14 | 7 |
| 3번째 ConvTranspose2d | kernel 3 · stride 2 · padding 1 · output_padding 1 | 7 | 14 |
| 4번째 ConvTranspose2d | kernel 3 · stride 2 · padding 1 · output_padding 1 | 14 | 28 |

In [8]:
# 검사 도구 — 고치지 않고 실행만 · 읽지 않아도 됨
def 크기_검사(표, model):
    설정 = (  # 표의 '설정' 칸: (층 종류, kernel, stride, padding, output_padding)
        ("Conv2d", 3, 2, 1, 0),
        ("Conv2d", 3, 2, 1, 0),
        ("ConvTranspose2d", 3, 2, 1, 1),
        ("ConvTranspose2d", 3, 2, 1, 1),
    )
    # 1) 표 자체가 표의 설정대로 센 크기와 같은지 (기준값은 찍지 않음)
    n, 기준 = 28, []
    for kind, k, s, p, op in 설정:
        n = (n + 2 * p - k) // s + 1 if kind == "Conv2d" else (n - 1) * s - 2 * p + k + op
        기준.append(n)
    틀린_줄 = [i + 1 for i in range(len(기준)) if i >= len(표) or 표[i] != 기준[i]]
    if 틀린_줄 or len(표) != len(기준):
        번호 = " · ".join(str(i) for i in 틀린_줄) or str(len(기준) + 1)
        print(f"[표 먼저] {번호}번째 줄의 '나오는 크기'가 표의 설정으로 센 값과 다름 → 칸 고르기 · 칸과 틈으로 다시 세고 표부터 고침")
        return
    # 2) 모델에 한 장을 넣어 층마다 찍힌 크기와 표 비교
    h = torch.zeros(1, 1, 28, 28)
    층, 찍힌 = [], []
    for layer in model:
        h = layer(h)
        if isinstance(layer, (nn.Conv2d, nn.ConvTranspose2d)):
            층.append(layer)
            print(f"{len(층)}번째 {type(layer).__name__:<16} 뒤", tuple(h.shape))
            찍힌.append(h.shape[-1])
    if len(찍힌) != len(표):
        print(f"Conv 층 수가 표와 다름 (표 {len(표)}개 · 찍힌 {len(찍힌)}개) → 층을 지우거나 더하지 않음")
        return
    처음 = next((i for i, (a, b) in enumerate(zip(표, 찍힌)) if a != b), None)
    for i, (a, b) in enumerate(zip(표, 찍힌)):
        print(f"{i + 1}번째 층  표 {a:>2}  찍힌 {b:>2}", "← 여기서 처음 다름" if i == 처음 else "")
    if 처음 is not None:
        print("표와 다른 층이 있음 → 처음 다른 층의 설정을 표와 비교해 한 곳만 고침")
        return
    # 3) 크기가 같아도 층 설정이 표의 설정과 같은지
    다른_층 = []
    for i, (layer, (kind, k, s, p, op)) in enumerate(zip(층, 설정)):
        실제 = (type(layer).__name__, layer.kernel_size[0], layer.stride[0], layer.padding[0],
              getattr(layer, "output_padding", (0,))[0], layer.dilation[0])
        if 실제 != (kind, k, s, p, op, 1):
            다른_층.append(i + 1)
    if 다른_층:
        print(f"크기는 모두 같지만 {' · '.join(str(i) for i in 다른_층)}번째 층의 설정이 표와 다름 → 표의 설정대로 고쳐야 통과")
        return
    print("통과 · 표 · 찍힌 크기 · 층 설정이 모두 같음")


print("검사 도구 준비 끝")

검사 도구 준비 끝


In [10]:
표 = [14, 7, 14, 28]   # 1·2·3번째 줄의 '나오는 크기'(들어가는 크기 아님) · 28 은 이미 적힘
# name '크기_검사' 에러가 나면 위 검사 도구 셀부터 실행

torch.manual_seed(0)
tiny = nn.Sequential(
    nn.Conv2d(1, 8, 3, stride=2, padding=1), nn.ReLU(),
    nn.Conv2d(8, 16, 3, stride=2, padding=1), nn.ReLU(),
    nn.ConvTranspose2d(16, 8, 3, stride=2, padding=1, output_padding=1), nn.ReLU(),
    nn.ConvTranspose2d(8, 1, 3, stride=2, padding=1, output_padding=1), nn.Sigmoid())

크기_검사(표, tiny)

1번째 Conv2d           뒤 (1, 8, 14, 14)
2번째 Conv2d           뒤 (1, 16, 7, 7)
3번째 ConvTranspose2d  뒤 (1, 8, 14, 14)
4번째 ConvTranspose2d  뒤 (1, 1, 28, 28)
1번째 층  표 14  찍힌 14 
2번째 층  표  7  찍힌  7 
3번째 층  표 14  찍힌 14 
4번째 층  표 28  찍힌 28 
통과 · 표 · 찍힌 크기 · 층 설정이 모두 같음
